# FOMC Shock and CRSP Event-Return Acquisition

This notebook freezes the monetary-policy shock, constructs exact CRSP trading windows, downloads only the required security dates, and creates firm-level market-adjusted event returns. All expensive queries are checkpointed by calendar year and can be resumed without repeating completed work.

## 1. Setup

The acquisition stage reuses the official SF Fed files saved by Notebook 00.

In [ ]:
from pathlib import Path
import getpass, importlib, json, sys, time
import numpy as np
import pandas as pd
from IPython.display import display

def locate_root():
    start=Path.cwd().resolve()
    for candidate in [start,*start.parents]:
        if (candidate/'full_replication_80q').exists() and (candidate/'economic_shock_sensitivity').exists(): return candidate
    raise FileNotFoundError('Could not locate Asset_Embeddings root.')
ROOT=locate_root(); MODULE=ROOT/'economic_shock_sensitivity'; RAW=MODULE/'data'/'raw'/'sf_fed'
EVENT_DATA=MODULE/'data'/'processed'/'fomc_event_returns'; YEARLY=EVENT_DATA/'security_yearly'
RESULTS=MODULE/'results'/'event_return_acquisition'; MANIFESTS=MODULE/'manifests'
for path in [EVENT_DATA,YEARLY,RESULTS,MANIFESTS]: path.mkdir(parents=True,exist_ok=True)
sys.path.insert(0,str(MODULE/'src'))
import rate_shock_acquisition as rsa
importlib.reload(rsa)  # Load helper edits even when the Jupyter kernel remains active.
first_queryable=rsa.first_queryable; build_event_trading_map=rsa.build_event_trading_map
sql_date_list=rsa.sql_date_list; aggregate_event_returns=rsa.aggregate_event_returns; file_sha256=rsa.file_sha256
print('Repository root:',ROOT)

## 2. Freeze the shock definition

The primary surprise is the SF Fed `STMT` measure: the first principal component of high-frequency money-market rate changes in the 30-minute FOMC statement window, normalized to move one-for-one with the daily one-year Treasury yield. Positive values therefore represent unexpected tightening. The combined statement-plus-press-conference `ME` measure is retained as a robustness variable.

In [ ]:
mps=pd.read_csv(RAW/'monetary_policy_surprises'/'mps.csv',parse_dates=['Date'])
events=(mps.loc[mps.Date.between('2005-01-01','2024-12-31'),['Date','STMT','ME']]
        .rename(columns={'Date':'event_date','STMT':'shock_stmt','ME':'shock_me'}).dropna(subset=['shock_stmt']).sort_values('event_date').reset_index(drop=True))
events['event_year']=events.event_date.dt.year
events['shock_stmt_standardized_full_sample']=(events.shock_stmt-events.shock_stmt.mean())/events.shock_stmt.std(ddof=1)
events.to_parquet(EVENT_DATA/'fomc_surprises_2005_2024.parquet',index=False)
shock_contract={'source':'Federal Reserve Bank of San Francisco USMPD','source_file_sha256':file_sha256(RAW/'monetary_policy_surprises'/'mps.csv'),'primary_column':'STMT','positive_sign':'unexpected monetary tightening / higher one-year Treasury yield','units':'percentage points','primary_window':'30-minute FOMC statement window','robustness_column':'ME','sample_start':'2005-01-01','sample_end':'2024-12-31','events':len(events),'full_sample_standardization_only_for_descriptive_use':True}
(MANIFESTS/'fomc_shock_contract_v1.json').write_text(json.dumps(shock_contract,indent=2,sort_keys=True)+'\n')
display(events.describe(include='all'))
display(events.groupby('event_year').size().rename('events').reset_index())

## 3. Establish an explicit WRDS connection

Enter credentials when prompted. They are held only in memory and are never printed or saved. If all downstream checkpoints already exist, this connection can be skipped by setting `CONNECT_TO_WRDS=False`.

In [ ]:
CONNECT_TO_WRDS=True
if CONNECT_TO_WRDS:
    import wrds
    username=input('WRDS username: ').strip(); password=getpass.getpass('WRDS password: ')
    if 'db' in globals():
        try: db.close()
        except Exception: pass
    db=wrds.Connection(wrds_username=username,wrds_password=password,autoconnect=False,verbose=True,
        wrds_connect_args={'sslmode':'require','connect_timeout':30,'application_name':'rate_shock_event_acquisition'})
    db.connect(); display(db.raw_sql('select 1 as connection_test'))
    print('WRDS connection is active and queryable.')
else: print('WRDS connection skipped; downstream cells require complete checkpoints.')

## 4. Download the CRSP trading calendar and market return

The small daily-index table determines the preceding and following trading day for every FOMC event. A failed table probe is rolled back before trying the next CRSP schema.

In [ ]:
market_path=EVENT_DATA/'crsp_market_calendar_2004_2025.parquet'
if market_path.exists():
    market=pd.read_parquet(market_path); market_source='local_checkpoint'
else:
    if 'db' not in globals(): raise ConnectionError('Run Section 3 or provide the market checkpoint.')
    market_candidates=[
      {'name':'crsp_a_indexes.dsi','probe':'select date, vwretd from crsp_a_indexes.dsi limit 1'},
      {'name':'crsp.dsi','probe':'select date, vwretd from crsp.dsi limit 1'},
      {'name':'crsp_a_stock.dsi','probe':'select date, vwretd from crsp_a_stock.dsi limit 1'}]
    selected=first_queryable(db,market_candidates); market_source=selected['name']
    market=db.raw_sql(f"select date, vwretd as market_return from {market_source} where date between '2004-12-01' and '2025-01-31' order by date",date_cols=['date'])
    market.to_parquet(market_path,index=False)
market['date']=pd.to_datetime(market.date); market['market_return']=pd.to_numeric(market.market_return,errors='coerce')
event_map=build_event_trading_map(events,market.date)
event_map.to_parquet(EVENT_DATA/'fomc_event_trading_date_map.parquet',index=False)
display(pd.DataFrame([{'market_source':market_source,'market_days':len(market),'first_date':market.date.min(),'last_date':market.date.max(),'events':events.event_date.nunique(),'required_security_dates':event_map.trading_date.nunique(),'nontrading_announcements':event_map.loc[event_map.nontrading_announcement,'event_date'].nunique()}]))
display(event_map.loc[event_map.nontrading_announcement].drop_duplicates('event_date')[['event_date','effective_event_date','calendar_days_to_market_open']])
display(event_map.head(9))

## 5. Select a point-in-time CRSP daily-security source

Modern CRSP daily v2 tables carry PERMCO directly. Legacy fallbacks join the names history on each observation date, which preserves point-in-time firm identity. This cell probes only one row and records the selected specification.

In [ ]:
security_candidates=[
 {'name':'crsp_a_stock.wrds_dsfv2_query','probe':'select permno, permco, dlycaldt, dlyret, dlyprc, shrout from crsp_a_stock.wrds_dsfv2_query limit 1','mode':'direct','table':'crsp_a_stock.wrds_dsfv2_query','date':'dlycaldt','ret':'dlyret','prc':'dlyprc','shrout':'shrout'},
 {'name':'crsp_a_stock.dsf_v2','probe':'select permno, permco, dlycaldt, dlyret, dlyprc, shrout from crsp_a_stock.dsf_v2 limit 1','mode':'direct','table':'crsp_a_stock.dsf_v2','date':'dlycaldt','ret':'dlyret','prc':'dlyprc','shrout':'shrout'},
 {'name':'crsp_a_stock.dsf + stocknames','probe':'select d.permno,n.permco,d.date,d.ret,d.prc,d.shrout from crsp_a_stock.dsf d join crsp_a_stock.stocknames n on d.permno=n.permno and d.date between n.namedt and n.nameenddt limit 1','mode':'legacy','daily':'crsp_a_stock.dsf','names':'crsp_a_stock.stocknames'},
 {'name':'crsp.dsf + stocknames','probe':'select d.permno,n.permco,d.date,d.ret,d.prc,d.shrout from crsp.dsf d join crsp.stocknames n on d.permno=n.permno and d.date between n.namedt and n.nameenddt limit 1','mode':'legacy','daily':'crsp.dsf','names':'crsp.stocknames'}]
if 'db' not in globals() and not all((YEARLY/f'crsp_event_security_{year}.parquet').exists() for year in range(2005,2025)):
    raise ConnectionError('Security checkpoints are incomplete; run Section 3.')
security_spec=first_queryable(db,security_candidates) if 'db' in globals() else {'name':'local_checkpoints','mode':'checkpoint'}
display(pd.DataFrame([{k:v for k,v in security_spec.items() if k!='probe'}]))

## 6. Download exact event dates with yearly checkpoints

Only the `−1`, `0`, and `+1` trading dates are requested. Completed years are loaded instead of queried again. The progress table reports rows, firms, file size, and elapsed time for each year.

In [ ]:
year_audit=[]; yearly_frames=[]
for year in range(2005,2025):
    started=time.perf_counter(); path=YEARLY/f'crsp_event_security_{year}.parquet'
    required=event_map.loc[event_map.event_date.dt.year.eq(year),'trading_date'].drop_duplicates()
    if path.exists(): frame=pd.read_parquet(path); status='checkpoint'
    else:
        dates_sql=sql_date_list(required)
        if security_spec['mode']=='direct':
            s=security_spec
            query=f"select permno,permco,{s['date']} as date,{s['ret']} as ret,{s['prc']} as prc,{s['shrout']} as shrout from {s['table']} where {s['date']} in ({dates_sql})"
        else:
            s=security_spec
            query=f"select distinct d.permno,n.permco,d.date,d.ret,d.prc,d.shrout from {s['daily']} d join {s['names']} n on d.permno=n.permno and d.date between n.namedt and n.nameenddt where d.date in ({dates_sql}) and n.shrcd in (10,11)"
        frame=db.raw_sql(query,date_cols=['date']); frame.to_parquet(path,index=False); status='downloaded'
    yearly_frames.append(frame)
    record={'year':year,'status':status,'required_dates':len(required),'rows':len(frame),'permnos':frame.permno.nunique(),'permcos':frame.permco.nunique(),'file_mb':path.stat().st_size/2**20,'seconds':time.perf_counter()-started}
    year_audit.append(record); print(f"{year}: {status}, {len(frame):,} rows, {record['seconds']:.1f}s")
security_audit=pd.DataFrame(year_audit); security_audit.to_parquet(RESULTS/'yearly_security_acquisition_audit.parquet',index=False)
display(security_audit)

## 7. Validate and combine the immutable raw extract

This catches duplicate security-date records, missing required dates, invalid identifiers, and accidental observations outside the requested calendar before constructing returns.

In [ ]:
security=pd.concat(yearly_frames,ignore_index=True); security['date']=pd.to_datetime(security.date)
for column in ['permno','permco']: security[column]=pd.to_numeric(security[column],errors='coerce').astype('Int64')
duplicate_rows_before=int(security.duplicated(['permno','date']).sum())
duplicate_groups=security.loc[security.duplicated(['permno','date'],keep=False)].groupby(['permno','date'],dropna=False)
conflicting_duplicate_keys=0 if duplicate_rows_before==0 else int((duplicate_groups[['permco','ret','prc','shrout']].nunique(dropna=False).max(axis=1)>1).sum())
assert conflicting_duplicate_keys==0, 'At least one duplicate PERMNO-date key contains conflicting CRSP values.'
# WRDS can return repeated, byte-equivalent rows. Preserve yearly raw checkpoints and deduplicate only this analytical copy.
security=security.drop_duplicates(['permno','date'],keep='first').reset_index(drop=True)
required_dates=set(pd.to_datetime(event_map.trading_date)); observed_dates=set(pd.to_datetime(security.date))
raw_audit=pd.DataFrame([{'raw_rows_before_deduplication':len(security)+duplicate_rows_before,'exact_duplicate_rows_removed':duplicate_rows_before,'conflicting_duplicate_keys':conflicting_duplicate_keys,'analytical_rows':len(security),'permnos':security.permno.nunique(),'permcos':security.permco.nunique(),'dates':security.date.nunique(),'required_dates':len(required_dates),'missing_required_dates':len(required_dates-observed_dates),'unexpected_dates':len(observed_dates-required_dates),'duplicate_security_dates_after_cleaning':int(security.duplicated(['permno','date']).sum()),'missing_permco_rows':int(security.permco.isna().sum())}])
raw_audit.to_parquet(RESULTS/'combined_raw_security_audit.parquet',index=False); display(raw_audit)
assert not (required_dates-observed_dates), 'At least one required CRSP trading date is absent.'
assert not (observed_dates-required_dates), 'The extract contains non-event-window dates.'
assert not security.duplicated(['permno','date']).any(), 'Duplicate PERMNO-date rows remain after exact deduplication.'

## 8. Construct PERMCO event returns

Security returns are compounded within each window and multiple securities are aggregated using market capitalization measured on trading day −1. Market-adjusted returns subtract the CRSP value-weighted market return over the same window.

In [ ]:
firm_events,security_events=aggregate_event_returns(security,event_map,market)
firm_path=EVENT_DATA/'firm_fomc_event_returns_2005_2024.parquet'; security_event_path=EVENT_DATA/'security_fomc_event_returns_2005_2024.parquet'
firm_events.to_parquet(firm_path,index=False); security_events.to_parquet(security_event_path,index=False)
firm_audit=pd.DataFrame([{'events':firm_events.event_date.nunique(),'firm_event_rows':len(firm_events),'firms':firm_events.permco.nunique(),'median_firms_per_event':firm_events.groupby('event_date').permco.nunique().median(),'valid_abnormal_00':firm_events.abnormal_return_00.notna().sum(),'valid_abnormal_01':firm_events.abnormal_return_01.notna().sum(),'multi_security_firm_events':int(firm_events.underlying_permnos.gt(1).sum()),'stored_mb':firm_path.stat().st_size/2**20}])
firm_audit.to_parquet(RESULTS/'firm_event_return_audit.parquet',index=False); display(firm_audit)
display(firm_events[['firm_return_00','market_return_00','abnormal_return_00','firm_return_01','market_return_01','abnormal_return_01','lag_market_equity']].describe(percentiles=[.01,.1,.5,.9,.99]))

## 9. Coverage against the embedding universe

Each event is mapped to the preceding quarter's embedding. This audit measures usable overlap without fitting the historical sensitivity target.

In [ ]:
coverage=[]
for event_date,group in firm_events.groupby('event_date'):
    event_quarter=pd.Period(event_date,freq='Q'); embedding_label=str(event_quarter-1).lower()
    path=ROOT/'full_replication_80q'/'embeddings'/embedding_label/'rs_binary'/'asset_embeddings_normalized_labeled.parquet'
    embedded=pd.read_parquet(path,columns=['permco']) if path.exists() else pd.DataFrame({'permco':[]})
    valid=group[group.abnormal_return_00.notna()]
    overlap=int(valid.permco.isin(embedded.permco).sum()); embedded_count=int(embedded.permco.nunique())
    coverage.append({'event_date':event_date,'event_quarter':str(event_quarter).lower(),'embedding_quarter':embedding_label,'valid_return_firms':valid.permco.nunique(),'embedded_firms':embedded_count,'overlap_firms':overlap,'share_of_crsp_firms_in_embedding':overlap/len(valid) if len(valid) else np.nan,'embedded_firm_return_coverage':overlap/embedded_count if embedded_count else np.nan})
coverage=pd.DataFrame(coverage); coverage.to_parquet(RESULTS/'event_embedding_coverage.parquet',index=False)
display(coverage.describe(include='all'))
display(coverage.tail(12))

## 10. Acquisition gate

The acquisition stage is complete only if all 172 shocks have event returns, all required trading dates were observed, and embedding overlap is adequate. The next notebook will estimate rolling five-year firm sensitivities using prior events only.

In [ ]:
gate=pd.DataFrame([
 {'test':'172 FOMC events represented','value':firm_events.event_date.nunique(),'passed':firm_events.event_date.nunique()==172},
 {'test':'all trading dates represented','value':len(required_dates-observed_dates),'passed':len(required_dates-observed_dates)==0},
 {'test':'no duplicate PERMNO-date rows','value':int(security.duplicated(['permno','date']).sum()),'passed':not security.duplicated(['permno','date']).any()},
 {'test':'median embedded-firm return coverage >= 90% from 2010','value':coverage.loc[coverage.event_date.ge('2010-01-01'),'embedded_firm_return_coverage'].median(),'passed':coverage.loc[coverage.event_date.ge('2010-01-01'),'embedded_firm_return_coverage'].median()>=.90},
])
gate.to_parquet(RESULTS/'acquisition_gate.parquet',index=False); display(gate)
print('Firm-event panel:',firm_path)
print('STOP: inspect all acquisition diagnostics before estimating rate sensitivities.')